In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/pacientes.csv"

In [0]:
pacientes_schema = StructType(fields=[
    StructField("id_paciente", IntegerType(), True),
    StructField("nombre_completo", StringType(), True),
    StructField("sexo", StringType(), True),
    StructField("edad", IntegerType(), True),
    StructField("fecha_nacimiento", StringType(), True),
    StructField("distrito", StringType(), True),
    StructField("tipo_seguro", StringType(), True)
])

In [0]:
df_pacientes_read = spark.read.option('header', True)\
                              .schema(pacientes_schema)\
                              .csv(ruta)

In [0]:
df_pacientes_dt = df_pacientes_read.withColumn("fecha_nacimiento", to_date(col("fecha_nacimiento"), "yyyy-MM-dd"))

In [0]:
df_pacientes_ingestion = df_pacientes_dt.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_pacientes_final = df_pacientes_ingestion.select(
    "id_paciente",
    "nombre_completo",
    "sexo",
    "edad",
    "fecha_nacimiento",
    "distrito",
    "tipo_seguro",
    "INGESTION_DATE"
)

In [0]:
df_pacientes_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.pacientes")